<a href="https://colab.research.google.com/github/Eva360563/distributed-architecture/blob/main/Lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
#Lab7
from pyspark import SparkContext, SparkConf
from datetime import datetime
conf = SparkConf().setAppName("Application name")
sc = SparkContext.getOrCreate(conf=conf)

register = sc.textFile("registerSample.csv")
register.take(5)
#stationID \ ttimestamp \ tusedslots\ tfreeslots

['station\ttimestamp\tused_slots\tfree_slots',
 '1\t2008-05-15 12:01:00\t0\t18',
 '1\t2008-05-15 12:02:00\t0\t18',
 '1\t2008-05-15 12:04:00\t0\t18',
 '1\t2008-05-15 12:06:00\t0\t18']

In [28]:
station = sc.textFile("stations.csv")
# stationID \ tlongitude \ tlatitude \tname

In [29]:
def cleanData(line):
  if line.startswith('s'):
    return False
  else:
    fields = line.split("\t")
    usedslots= int(fields[2])
    freeslot = int(fields[3])
    if freeslot != 0 or usedslots != 0:
      return True
    else:
      return False

In [30]:
filtered = register.filter(cleanData)

In [32]:
#devo riscrivere il mio rdd con giorno della settimana e ora
def checkFull(line):
  fields= line.split("\t")
  stationId = fields[0]
  freeSlots = int(fields[3])
  timestamp = fields[1]

  datetimeobject= datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
  dayOfWeek= datetimeobject.strftime("%a")
  hour= datetimeobject.hour

  if freeSlots == 0:
    counttotreading = (1,1)
  else:
    counttotreading = (1,0)
  return ((stationId, dayOfWeek, hour), counttotreading)


In [34]:
stationdayweek = filtered.map(checkFull)
stationdayweek.take(5)

[(('1', 'Thu', 12), (1, 0)),
 (('1', 'Thu', 12), (1, 0)),
 (('1', 'Thu', 12), (1, 0)),
 (('1', 'Thu', 12), (1, 0)),
 (('1', 'Thu', 12), (1, 0))]

In [43]:
reduced= stationdayweek.reduceByKey(lambda s1, s2: (s1[0] + s2[0], s1[1] + s2[1] ))
relation = reduced.mapValues(lambda s: s[1]/s[0])
criticalstation= relation.filter(lambda s: s[1] >= 0.4)
criticalstation.collect()

[(('1', 'Thu', 1), 0.4329608938547486),
 (('1', 'Sun', 4), 0.403899721448468),
 (('1', 'Thu', 0), 0.4581005586592179)]

In [63]:
stationTimeslot= criticalstation.map(lambda s: (s[0][0], (s[0][1], s[0][2], s[1])))
stationTimeslot.collect()

[('1', ('Thu', 1, 0.4329608938547486)),
 ('1', ('Sun', 4, 0.403899721448468)),
 ('1', ('Thu', 0, 0.4581005586592179))]

In [64]:
#definisco la funzione per mettere in ordine
def comparecriticality(timeslot1, timeslot2):
  weekday1 = timeslot1[0]
  weekday2 = timeslot2[0]

  hour1 = timeslot1[1]
  hour2 = timeslot2[1]

  crit1 = timeslot1[2]
  crit2 = timeslot2[2]

  if (crit1> crit2) or(crit1== crit2 and hour1< hour2 ) or (crit1== crit2 and hour1== hour2 and weekday1< weekday2):
    return timeslot1
  else:
    return timeslot2

In [65]:
result = stationTimeslot.reduceByKey(comparecriticality)
result.collect()

[('1', ('Thu', 0, 0.4581005586592179))]

In [66]:
def extractStationLongLat(line):
    fields = line.split("\t")

    return (fields[0], (fields[1] ,fields[2]) )

In [67]:
stationlocation = station.map(extractStationLongLat)

In [68]:
resultlocation = result.join(stationlocation)

In [69]:
def formatKMLMarker(pair):
    # input
    # (stationId, ( (weekday, hour, criticality), (long, lat) ) )
    stationId = pair[0]

    weekday = pair[1][0][0]
    hour = pair[1][0][1]
    criticality = pair[1][0][2]
    coordinates = pair[1][1][0]+","+pair[1][1][1]

    result = "<Placemark><name>" + stationId + "</name>" + "<ExtendedData>"\
    + "<Data name=\"DayWeek\"><value>" + weekday + "</value></Data>"\
    + "<Data name=\"Hour\"><value>" + str(hour) + "</value></Data>"\
    + "<Data name=\"Criticality\"><value>" + str(criticality) + "</value></Data>"\
    + "</ExtendedData>" + "<Point>" + "<coordinates>" + coordinates + "</coordinates>"\
    + "</Point>" + "</Placemark>"

    return result

In [70]:
resultKML = resultlocation.map(formatKMLMarker)

In [71]:
resultKML.coalesce(1).collect()

['<Placemark><name>1</name><ExtendedData><Data name="DayWeek"><value>Thu</value></Data><Data name="Hour"><value>0</value></Data><Data name="Criticality"><value>0.4581005586592179</value></Data></ExtendedData><Point><coordinates>2.180019,41.397978</coordinates></Point></Placemark>']

In [73]:
outputPath = "out_Lab7_Sample_04/" # args[3]
resultKML.coalesce(1).saveAsTextFile(outputPath)